# WI-91 探针夹具（**不是教程，不是交付物**）

这是里程碑 WI-91（子项目 A2：给镜像里加一条**真能交互执行**的 Scala Jupyter kernel）三条候选路径
（Almond → Toree → 自包 `IMain`）**共用的判据本体**。执行器是 `server/test/notebooks/scalaKernelProbe.ts`，
调用形状是 `probeScalaKernel('<待评的 kernel 名>')` —— kernel 名由参数给，本文件的
`metadata.kernelspec.name` 只是占位（实测 `--ExecutePreprocessor.kernel_name` 会覆盖它，且**输出的 notebook 里那一格不改写**，
所以"真跑的是哪条 kernel"要看 `metadata.language_info`，见执行器注释）。

## 三个 code cell 各判什么

| cell | 判的东西 | 谁会被这一格拦下 |
| --- | --- | --- |
| 1 | `val doubled = 6 * 7` 能编译、能执行、`println` 打得出来 | 连 Scala 都不认的 kernel（镜像里的 `python3` 就是反面对照） |
| 2 | **跨 cell 状态存活**：`alive=42` | "编译一段跑一段"那种替代形状 —— 它每一段都是全新的 `IMain`，第二段就会 `not found: value doubled` |
| 3 | 起得来一个**活的 `SparkSession`**，`spark.range(6).count()` 真是 6 行 | 只装了 Scala 编译器、没把 `/opt/spark/jars` 挂上 classpath 的那条装配 |

## 为什么"跨 cell 存活"才是这一档的意义

没有它，Scala 侧的 notebook 就只是一个**分段编译器**：读者拿不到"改一行、重跑一次"的那个循环 ——
而那个循环正是 notebook 相对于"交一段代码等判题"的全部价值。今天镜像里只有 `arena-pyspark`
满足它（Python 的 IPython kernel 天生有活状态），Scala 侧只能靠 `spark-worker.scala` 那种
"编译一段跑一段"的形状，于是 Scala 教程永远写不成。这一档要判的就是**那条循环到底有没有真的活过来**。

⇒ 判据因此取的是 **stdout 里出现 `alive=42` 这一行**，**不是**"第二个 cell 没报错"：
报错与否取决于 kernel 怎么实现（有的 kernel 把未定义符号吞成一条 warning、有的把整段编译失败报成
一条 `output_type=error`），而 `alive=42` 只取决于**状态到底活没活**。

## 两条刻意的"不许冒充"

- cell 3 打的是 `rows=6`（`range(6)` 的**行数**），与 `content/notebooks/00-smoke-pyspark.ipynb` 打的
  `rows 15`（同一表达式的 id **之和**）**故意不是同一个串**，而且两行 marker 都取"整行相等"而不是子串
  —— 否则 `rows=60` 也会 `includes('rows=6')`。
- 两条**反面**自测各判一件事（Task 1 Step 4 的实测读数在 `task-1-report.md` 里）：
  ① 本夹具跑在镜像自带的 `python3` 上 ⇒ 三个 code cell 全报 Python `SyntaxError`，三条判据全红；
  ② **反向指认**：把执行器指到 `content/notebooks/00-smoke-pyspark.ipynb`（一篇已知跑得绿的 Python Spark 教程）上跑
  ⇒ `errors` 为空而 `stateSurvived` / `sparkSession` 仍为 false。②判的是"`ok` 不等于'没报错'"这件事本身。
- 最后一个 code cell 带 `spark.stop()`：不留活着的 Spark JVM 给下一次探针，也不给隔壁那些
  正在容器里跑教程的档位添内存账（同 `tutorials.test.ts` 那条"最后一个 code cell 必须 spark.stop()"的理由）。

## 已知边界（写在这里，免得下一个人以为这里还留着牙）

- cell 3 照简报逐字写，**不带** `.master(...)`：如果某条 kernel 的装配里没有 `spark.master` 默认值
  （镜像里 `/opt/spark/conf/` 目录根本不存在，实测没有 `spark-defaults.conf`），它会在这里报
  `A master URL must be set in your configuration` —— 那是**那条 kernel 的装配账**（要么 kernelspec 的 `env`
  里给 `PYSPARK_SUBMIT_ARGS` 那种同款，要么在 cell 里补 `.master("local[2]")`），不是判据写错了。
- 本夹具只看 `stream`/`stdout` 那几行与 `output_type=error` 的条目（`notebook-evidence.ts` 那两个工具能取的东西）。
  若某条 kernel 把 `println` 落成 `execute_result` 而不是 `stream`，这里会**假红** —— 那属于"探针看不见"，
  要按执行器顶部那段说出去，别去改判据。

In [ ]:
val doubled = 6 * 7
println(s"doubled=$doubled")

In [ ]:
println(s"alive=$doubled")

In [ ]:
val spark = org.apache.spark.sql.SparkSession.builder().appName("a2-probe").getOrCreate()
println(s"rows=${spark.range(6).count()}")   // 6，不是 15：这一条只数行数
spark.stop()